# Personalisierung des besten Models (RF)

- Baseline Normalisierung
- Kalibrierung auf wenigen Trainingssamples der Zielperson
- Voll personalisiert, nur auf Daten der Zielperson 

## Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scripts.myml import loso_binary_nested_cv, loso_binary_calibrated_nested_cv, loso_binary_fully_personalized_nested_cv
from scripts.feature_engeneering import subject_baseline_normalization
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier, XGBRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

## Daten laden

In [2]:
DATASET_PATH = Path("dataset/np-dataset")
X = np.load(DATASET_PATH / "X.npy")
X_catch22 = np.load(DATASET_PATH / "X_catch22.npy")
X_catch22_feature_names = np.load(DATASET_PATH / "feature_names.npy")
y_heater = np.load(DATASET_PATH / "y_heater.npy")
subjects = np.load(DATASET_PATH / "subjects.npy")
y = np.argmax(y_heater, axis=1)

# EMG sensor bei 42 war defekt
valid = subjects != 42
X_catch22 = X_catch22[valid]
y = y[valid]
subjects = subjects[valid]

In [3]:
print(f"X shape : {X_catch22.shape}")
print(f"y shape : {y.shape}   classes: {np.unique(y).tolist()}")
print(f"subjects : {np.unique(subjects).shape[0]} unique subjects")
print(f"Class counts : { {c: int((y==c).sum()) for c in range(6)} }")

X shape : (2447, 174)
y shape : (2447,)   classes: [0, 1, 2, 3, 4, 5]
subjects : 51 unique subjects
Class counts : {0: 408, 1: 408, 2: 408, 3: 407, 4: 408, 5: 408}


## Hyperparameter




In [4]:
# Random Forest
PARAM_GRID_RF = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

INNER_N_SPLITS = 5
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Feature-Matrixes

In [5]:
# without normalization
X_catch22 = X_catch22.copy()

# with subject-specific normalization
baseline_class = 0 # samples with no stimulus 
X_catch22_normalized = subject_baseline_normalization(X_catch22, y, subjects, baseline_class)

print(np.nanmean(X_catch22))
print(np.nanmean(X_catch22_normalized))

40.55045877855115
1.289692166951647


## Setup for binary classification

- without baseline norm

In [6]:
class_comparison = (0, 5)
mask = (y==class_comparison[0]) | (y==class_comparison[1])
X_filtered = X_catch22[mask]
y_filtered = y[mask]
group_labels = subjects[mask]
y_binary = (y_filtered == class_comparison[1]).astype(int)

- with baseline norm

In [7]:
class_comparison = (0, 5)
mask = (y==class_comparison[0]) | (y==class_comparison[1])
X_filtered_normalized = X_catch22_normalized[mask]
y_filtered = y[mask]
group_labels = subjects[mask]
y_binary = (y_filtered == class_comparison[1]).astype(int)

# RF subject independent
- without baseline norm

In [8]:
# Random Forest
rf = RandomForestClassifier(random_state=RANDOM_STATE)
rf_subject_independent = loso_binary_nested_cv(
    X_filtered,
    y_binary,
    group_labels,
    rf,
    PARAM_GRID_RF,
    model_type="classifier"
)
print(f"    Accuracy : {rf_subject_independent['accuracy']:.3f} ± {rf_subject_independent['accuracy_std']:.3f}")
print(f"    AUC      : {rf_subject_independent['auc']:.3f} ± {rf_subject_independent['auc_std']:.3f}")
print(f"    F1       : {rf_subject_independent['f1']:.3f} ± {rf_subject_independent['f1_std']:.3f}")
print(f"    Sensitivity : {rf_subject_independent['sensitivity']:.3f} ± {rf_subject_independent['sensitivity_std']:.3f}")
print(f"    Specificity : {rf_subject_independent['specificity']:.3f} ± {rf_subject_independent['specificity_std']:.3f}")

    Accuracy : 0.917 ± 0.081
    AUC      : 0.976 ± 0.048
    F1       : 0.913 ± 0.090
    Sensitivity : 0.906 ± 0.135
    Specificity : 0.926 ± 0.122


## RF subject independent
- with baseline norm

In [9]:
# Random Forest
rf = RandomForestClassifier(random_state=RANDOM_STATE)
rf_subject_independent_normalized = loso_binary_nested_cv(
    X_filtered_normalized,
    y_binary,
    group_labels,
    rf,
    PARAM_GRID_RF,
    model_type="classifier"
)
print(f"    Accuracy : {rf_subject_independent_normalized['accuracy']:.3f} ± {rf_subject_independent_normalized['accuracy_std']:.3f}")
print(f"    AUC      : {rf_subject_independent_normalized['auc']:.3f} ± {rf_subject_independent_normalized['auc_std']:.3f}")
print(f"    F1       : {rf_subject_independent_normalized['f1']:.3f} ± {rf_subject_independent_normalized['f1_std']:.3f}")
print(f"    Sensitivity : {rf_subject_independent_normalized['sensitivity']:.3f} ± {rf_subject_independent_normalized['sensitivity_std']:.3f}")
print(f"    Specificity : {rf_subject_independent_normalized['specificity']:.3f} ± {rf_subject_independent_normalized['specificity_std']:.3f}")

    Accuracy : 0.944 ± 0.068
    AUC      : 0.987 ± 0.025
    F1       : 0.940 ± 0.075
    Sensitivity : 0.928 ± 0.108
    Specificity : 0.958 ± 0.064


## RF calibrated
- weighted: the k samples from patient have the same weight as all other trainng samples together
- uses baseline norm

In [11]:
calibration_comparison = [2, 4, 6, 8]

rf = RandomForestClassifier(random_state=RANDOM_STATE)
rf_calibrated_k = {}

for k in calibration_comparison:
    np.random.seed(RANDOM_STATE)
    print (f"Calibration samples per class: {k//2} (total {k})")
    print(f"Training model: {rf}...")
    rf_calibrated = loso_binary_calibrated_nested_cv(
            X_filtered_normalized, 
            y_binary, 
            group_labels, 
            rf,
            PARAM_GRID_RF, 
            k,
            balance=True
    )

    rf_calibrated_k[k] = rf_calibrated

    print(f"    Accuracy : {rf_calibrated['accuracy']:.3f} ± {rf_calibrated['accuracy_std']:.3f}")
    print(f"    AUC      : {rf_calibrated['auc']:.3f} ± {rf_calibrated['auc_std']:.3f}")
    print(f"    F1       : {rf_calibrated['f1']:.3f} ± {rf_calibrated['f1_std']:.3f}")
    print(f"    Sensitivity : {rf_calibrated['sensitivity']:.3f} ± {rf_calibrated['sensitivity_std']:.3f}")
    print(f"    Specificity : {rf_calibrated['specificity']:.3f} ± {rf_calibrated['specificity_std']:.3f}")

Calibration samples per class: 1 (total 2)
Training model: RandomForestClassifier(random_state=42)...


    Accuracy : 0.945 ± 0.064
    AUC      : 0.990 ± 0.021
    F1       : 0.943 ± 0.068
    Sensitivity : 0.933 ± 0.100
    Specificity : 0.958 ± 0.065
Calibration samples per class: 2 (total 4)
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.948 ± 0.069
    AUC      : 0.988 ± 0.026
    F1       : 0.947 ± 0.071
    Sensitivity : 0.945 ± 0.096
    Specificity : 0.951 ± 0.083
Calibration samples per class: 3 (total 6)
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.955 ± 0.066
    AUC      : 0.993 ± 0.022
    F1       : 0.952 ± 0.072
    Sensitivity : 0.942 ± 0.110
    Specificity : 0.969 ± 0.073
Calibration samples per class: 4 (total 8)
Training model: RandomForestClassifier(random_state=42)...
    Accuracy : 0.966 ± 0.061
    AUC      : 0.995 ± 0.021
    F1       : 0.965 ± 0.062
    Sensitivity : 0.962 ± 0.089
    Specificity : 0.971 ± 0.081


## RF fully personalized

In [ ]:
rf = RandomForestClassifier(random_state=RANDOM_STATE)
rf_personalized = loso_binary_fully_personalized_nested_cv(
        X_filtered_normalized, 
        y_binary, 
        group_labels, 
        rf, 
        PARAM_GRID_RF, 
        'classifier'
)

print(f"    Accuracy : {rf_personalized['accuracy']:.3f} ± {rf_personalized['accuracy_std']:.3f}")
print(f"    AUC      : {rf_personalized['auc']:.3f} ± {rf_personalized['auc_std']:.3f}")
print(f"    F1       : {rf_personalized['f1']:.3f} ± {rf_personalized['f1_std']:.3f}")
print(f"    Sensitivity : {rf_personalized['sensitivity']:.3f} ± {rf_personalized['sensitivity_std']:.3f}")
print(f"    Specificity : {rf_personalized['specificity']:.3f} ± {rf_personalized['specificity_std']:.3f}")

# Vergleich

In [ ]:
model_results = {
    'Subject-Independent': rf_subject_independent,
    'Subject-Independent Normalized': rf_subject_independent_normalized,
    'Calibrated (k=2)': rf_calibrated_k[2],
    'Calibrated (k=4)': rf_calibrated_k[4],
    'Calibrated (k=6)': rf_calibrated_k[6],
    'Calibrated (k=8)': rf_calibrated_k[8],
    'Fully Personalized': rf_personalized
}

## Verglechs Tabelle

In [ ]:
rows = []

for name, res in model_results.items():
    rows.append({
        'Model': name,
        'Accuracy': f"{res['accuracy']:.3f} ± {res['accuracy_std']:.3f}",
        'AUC': f"{res['auc']:.3f} ± {res['auc_std']:.3f}",
        'F1 Score': f"{res['f1']:.3f} ± {res['f1_std']:.3f}",
        'Sensitivity': f"{res['sensitivity']:.3f} ± {res['sensitivity_std']:.3f}",
        'Specificity': f"{res['specificity']:.3f} ± {res['specificity_std']:.3f}"
    })

comparison_df = pd.DataFrame(rows).set_index('Model')
comparison_df

,Accuracy,AUC,F1 Score,Sensitivity,Specificity
Model,,,,,
Subject-Independent,0.917 ± 0.081,0.976 ± 0.048,0.913 ± 0.090,0.906 ± 0.135,0.926 ± 0.122
Subject-Independent Normalized,0.944 ± 0.068,0.987 ± 0.025,0.940 ± 0.075,0.928 ± 0.108,0.958 ± 0.064
Calibrated (k=2),0.941 ± 0.070,0.988 ± 0.023,0.939 ± 0.073,0.924 ± 0.104,0.958 ± 0.076
Calibrated (k=4),0.951 ± 0.067,0.988 ± 0.033,0.947 ± 0.077,0.932 ± 0.119,0.971 ± 0.064
Calibrated (k=6),0.937 ± 0.095,0.985 ± 0.039,0.932 ± 0.108,0.924 ± 0.142,0.949 ± 0.104
Calibrated (k=8),0.943 ± 0.088,0.989 ± 0.042,0.938 ± 0.107,0.934 ± 0.141,0.951 ± 0.099
Fully Personalized,0.923 ± 0.158,0.981 ± 0.085,0.900 ± 0.231,0.904 ± 0.246,0.947 ± 0.193
